# Abu Qir Change Detection — Interactive Explorer

Yasser Aldegwy · Alexandria, Egypt · Google Earth Engine + geemap.

Run the cells in order. Edit `config.py` to change AOI / periods / parameters.

In [ ]:
# 1. Auth + init
import ee, geemap
import config
try:
    ee.Initialize(project=config.GEE_PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=config.GEE_PROJECT)
print('EE ready:', config.GEE_PROJECT)

In [ ]:
# 2. Load helpers from the main workflow
from abu_qir_workflow import (
    get_aoi, landsat_composite, s2_composite,
    add_indices_landsat, add_indices_s2,
    builtup_from_dynamic_world, builtup_from_rf,
    post_classification_change, extract_shoreline,
)
aoi = get_aoi()
print('AOI:', config.AOI_BBOX)

In [ ]:
# 3. Build two composites and show side-by-side
first = config.LANDSAT_PERIODS[0]
last  = config.LANDSAT_PERIODS[-1]
img_a = add_indices_landsat(landsat_composite(first[1], first[2], aoi))
img_b = add_indices_landsat(landsat_composite(last[1],  last[2],  aoi))

m = geemap.Map(center=[(config.AOI_BBOX[1]+config.AOI_BBOX[3])/2,
                       (config.AOI_BBOX[0]+config.AOI_BBOX[2])/2], zoom=12)
m.add_basemap('SATELLITE')
m.split_map(
    left_layer = geemap.ee_tile_layer(img_a, config.RGB_VIS_LANDSAT, f'Landsat {first[0]}'),
    right_layer= geemap.ee_tile_layer(img_b, config.RGB_VIS_LANDSAT, f'Landsat {last[0]}'))
m

In [ ]:
# 4. Built-up classification — Landsat era proxy vs Dynamic World
builtup_a = builtup_from_rf(img_a, aoi)
builtup_b = builtup_from_dynamic_world(last[1], last[2], aoi)
change = post_classification_change(builtup_a, builtup_b)

m2 = geemap.Map(center=[(config.AOI_BBOX[1]+config.AOI_BBOX[3])/2,
                        (config.AOI_BBOX[0]+config.AOI_BBOX[2])/2], zoom=12)
m2.add_basemap('SATELLITE')
m2.addLayer(change, {'min':0,'max':3,
    'palette':['#dddddd','#7f7f7f','#d7191c','#2c7bb6']},
    'Change class (gain=red, loss=blue)')
m2

In [ ]:
# 5. Shoreline overlay for the latest period
sl = extract_shoreline(img_b, aoi)
m3 = geemap.Map(center=[(config.AOI_BBOX[1]+config.AOI_BBOX[3])/2,
                        (config.AOI_BBOX[0]+config.AOI_BBOX[2])/2], zoom=12)
m3.add_basemap('SATELLITE')
m3.addLayer(sl.select('shoreline').selfMask(),
            {'palette':['#ffeb3b']}, f'Shoreline {last[0]}')
m3